# Verbalyze: Fine-Tune Indic Voice SLMs (Llama 3.2 3B / Qwen 2.5 3B) with QLoRA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ansh-rohilla/Verbalyze/blob/main/notebooks/train_indic_voice_slm.ipynb)
[![Hugging Face](https://img.shields.io/badge/🤗%20Hugging%20Face-Dataset-blue)](https://huggingface.co/datasets/ansh-rohilla/verbalyze-dialogues)

This notebook provides a complete 1-click training pipeline to fine-tune **Llama-3.2-3B-Instruct** or **Qwen-2.5-3B-Instruct** into a low-latency telephony voice agent on Google Colab (free T4 GPU supported).

### Highlights:
- **Dataset**: `ansh-rohilla/verbalyze-dialogues` (16,370 multi-turn voice conversations across 12 Indian languages)
- **Telephony Optimized**: 1024 token sequence length, conversational turn-taking, spoken fillers (*haan*, *hmm*), and native `disconnect_tool` function calling
- **Low Memory Footprint**: 4-bit NF4 Quantization (QLoRA) consuming < 6GB VRAM
- **Direct Hub Export**: Uploads your fine-tuned adapter directly to your Hugging Face profile.

## Step 1: Install Dependencies
Install Hugging Face TRL, PEFT, Transformers, BitsAndBytes, and Datasets.

In [ ]:
# Install fine-tuning packages
!pip install -q torch transformers datasets peft trl accelerate bitsandbytes huggingface_hub
!nvidia-smi

## Step 2: Load the Verbalyze Dataset from Hugging Face
Loads the 16,370 multi-turn telephony conversations across 12 Indian languages directly from the Hugging Face Hub.

In [ ]:
from datasets import load_dataset

DATASET_ID = "ansh-rohilla/verbalyze-dialogues"
print(f"Loading {DATASET_ID} from Hugging Face Hub...")
dataset = load_dataset(DATASET_ID)

print("\nDataset Summary:")
print(dataset)

sample = dataset["train"][0]
print(f"\nSample Conversation ID: {sample['id']}")
print(f"Total turns in sample: {len(sample['messages'])}")
for m in sample["messages"][:3]:
    print(f"  [{m['role'].upper()}]: {m['content'][:120]}...")

## Step 3: Model Setup & 4-bit Quantization
Select either `meta-llama/Llama-3.2-3B-Instruct` or `Qwen/Qwen2.5-3B-Instruct`.
We use 4-bit NormalFloat4 (NF4) quantization to fit the model easily inside a 16GB T4 GPU.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Select Base Model (Choose one):
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading base model: {MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# Configure LoRA Adapters targeting all linear projections
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 4: Fine-Tuning with TRL SFTTrainer
We train for 3 epochs with a cosine learning rate scheduler and gradient accumulation to simulate a batch size of 16.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "./verbalyze-voice-slm-3b"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    dataset_text_field="messages",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting QLoRA fine-tuning loop...")
trainer.train()

print(f"Saving trained adapter weights to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Fine-tuning complete!")

## Step 5: Test Telephony Voice Inference
Let's test the fine-tuned model on an outbound loan collection query to verify:
1. Spoken fillers (*"haan"*, *"hmm"*, *"ji"*)
2. Short conversational turn-taking (1-2 sentences)
3. Calling the `disconnect_tool` or `send_payment_link` on resolution.

In [ ]:
test_dialogue = [
    {
        "role": "system",
        "content": (
            "You are a friendly, polite collection agent representing Muthoot Fincorp. "
            "You are having a real phone call with a customer regarding their overdue EMI recovery of Rs. 5,420. "
            "Keep sentences SHORT (1-2 sentences max). Use natural fillers like 'haan', 'ji'. "
            "If payment link is requested, offer UPI link. If call is resolved, invoke disconnect_tool."
        )
    },
    {"role": "assistant", "content": "नमस्कार मिस्टर शर्मा, मैं मुथूट फिनकॉर्प से बोल रही हूँ। क्या आप आज अपनी ईएमआई जमा कर पाएंगे?"},
    {"role": "user", "content": "हाँ, आज ही कर देता हूँ। आप मुझे पेमेंट का लिंक भेज दीजिए।"}
]

inputs = tokenizer.apply_chat_template(
    test_dialogue,
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda" if torch.cuda.is_available() else "cpu")

outputs = model.generate(
    inputs,
    max_new_tokens=80,
    temperature=0.6,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("\n📞 Agent Generated Telephony Response:")
print(response)

## Step 6: Push Model Adapter to Hugging Face Hub 🤗
Publish your trained adapter weights to Hugging Face so you can deploy them in production or share them with the community.

In [ ]:
from huggingface_hub import login

# Authenticate with your HF Write token
login()

HUB_MODEL_ID = "ansh-rohilla/verbalyze-voice-llama3.2-3b-adapter"
print(f"Pushing adapter weights to https://huggingface.co/{HUB_MODEL_ID}...")

model.push_to_hub(HUB_MODEL_ID)
tokenizer.push_to_hub(HUB_MODEL_ID)
print(f"🎉 Model adapter successfully published to https://huggingface.co/{HUB_MODEL_ID}!")